In [ ]:
import random
import json
import re
import os
import copy
import asyncio
import pandas as pd
from pydantic import BaseModel, Field
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.utils.llm_requests_v3 import *
# from local_variables import phenomena_to_good_direction_verb_dict, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import EXPERIMENTS

system_prompt = EXPERIMENTS['evaluate_factuality_of_news_articles']['generate_articles']['system_prompt']
user_prompt_template = EXPERIMENTS['evaluate_factuality_of_news_articles']['generate_articles']['user_prompt_template']

In [ ]:
random.seed(42) # for reproducibility

# set model and model kwargs
model_name = "gpt-5.4"
# model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 


# make request to LLM to generate list of n views
topic = "The economic performance of the United States in 2024"
# political_bias_of_article = "right"
user_prompt = user_prompt_template.format(topic=topic)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)
#place list of views in pandas dataframe and save to csv
# df = pd.DataFrame([view.dict() for view in response.views])
# df.to_csv("./data/experimental_designs.csv", index=True)
# df


In [ ]:
async def generate_newspaper_articles(models, n, topics, outlets, system_prompt, user_prompt_template, custom_model_kwargs={}):
    tasks = []
    for i in range(n//2):  # We will generate 2 articles (left and right) for each topic
        topic = random.choice(topics)
        model_name = random.choice(models)
        model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
        user_prompt = user_prompt_template.format(topic=topic)
        payload = {
            "model_name": model_name,
            "system_prompt": system_prompt,
            "user_prompt": user_prompt,
            "topic": topic,
        }
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        tasks.append((payload, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (payload, _) in enumerate(tasks):
        response = results[idx]
        if isinstance(response, Exception):
            print(f"Exception for payload {payload}: {response}")
            continue
        payload["article"] = response
        for outlet_bias in ["left", "right"]:
            payload_copy = copy.deepcopy(payload)
            outlet = random.choice(outlets[outlet_bias])
            payload_copy["political_pole"] = outlet_bias
            payload_copy["outlet"] = outlet
            payloads.append(payload_copy)

    file_name = f"./data/articles.csv"
    df_experimental_designs = pd.DataFrame(payloads)
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    df_experimental_designs.to_csv(file_name, index=False)

    return payloads

topics = [
    "Climate change policy and emissions targets",
    "Immigration reform and border control",
    "Healthcare system funding and access",
    "Education curriculum standards and political influence",
    "Tax policy and wealth redistribution",
    "Housing affordability and zoning laws",
    "Police reform and public safety",
    "Freedom of speech and censorship laws",
    "Election integrity and voting systems",
    "Indigenous rights and treaty obligations",
    "Energy policy and renewable transition",
    "Inflation and cost-of-living crisis",
    "Minimum wage and labor rights",
    "Gender identity laws and public policy",
    "Foreign aid and international diplomacy",
    "Trade agreements and tariffs",
    "National security and surveillance",
    "Gun control legislation",
    "Public transportation funding",
    "Urban vs rural political divides",
    "Media bias and misinformation",
    "Corporate regulation and antitrust laws",
    "Environmental conservation vs economic growth",
    "Water rights and resource management",
    "Prison reform and incarceration rates",
    "Drug policy and legalization",
    "Mental health funding and services",
    "Religious freedom vs secular governance",
    "AI regulation and ethics",
    "Social welfare programs and benefits",
    "Aging population and pension systems",
    "Youth voting engagement",
    "Campaign finance laws",
    "Political lobbying and influence",
    "Digital privacy and data protection",
    "Cybersecurity and national defense",
    "Public broadcasting and government funding",
    "Infrastructure spending priorities",
    "Climate migration and refugee policy",
    "Food security and agricultural subsidies",
    "Fisheries regulation and sustainability",
    "Public health mandates and individual rights",
    "Pandemic preparedness and response",
    "Defense spending and military strategy",
    "Nuclear energy and safety concerns",
    "Space policy and militarization",
    "Regional autonomy and federalism",
    "Constitutional reform debates",
    "Judicial independence and court appointments",
    "Corruption and transparency in government",
    "Whistleblower protections",
    "Freedom of the press",
    "Internet regulation and net neutrality",
    "Gig economy and worker protections",
    "Automation and job displacement",
    "Economic inequality and social mobility",
    "Trade unions and collective bargaining",
    "Student debt and higher education costs",
    "Tourism policy and economic impact",
    "Cultural heritage and national identity",
    "Language policy and minority rights",
    "Public art funding controversies",
    "Urban development and gentrification",
    "Land use and environmental protection",
    "Mining policy and indigenous land rights",
    "Fishing quotas and international disputes",
    "Sanctions and geopolitical tensions",
    "Refugee resettlement programs",
    "Citizenship laws and naturalization",
    "Voting age debates",
    "Electoral system reform (e.g., proportional representation)",
    "Political polarization and partisanship",
    "Extremism and radicalization",
    "Hate speech laws",
    "Online platforms and political content moderation",
    "Government debt and fiscal responsibility",
    "Central bank independence",
    "Cryptocurrency regulation",
    "Universal basic income",
    "Climate litigation and legal accountability",
    "Environmental justice movements",
    "Public-private partnerships in infrastructure",
    "Disaster response funding",
    "Rural healthcare access",
    "Urban crime policy",
    "Border disputes and territorial claims",
    "Maritime law and exclusive economic zones",
    "International climate agreements",
    "Defense alliances and military pacts",
    "Peacekeeping missions",
    "Trade war impacts",
    "Human rights violations abroad",
    "Election interference and foreign influence",
    "National identity and immigration narratives",
    "Public sector strikes",
    "Housing market regulation",
    "Rent control policies",
    "Tax havens and offshore finance",
    "Corporate social responsibility and regulation"
]

outlets ={ #https://www.allsides.com/media-bias/media-bias-chart

        'left': [
            "AP",
            "The Atlantic",
            "The Guardian",
            "HuffPost",
            "The New Yorker",
            "Vox",
            "MSNBC",
            "The Nation",
            "Slate",
            "Daily Beast"
        ],

        'right': [
            "Fox News",
            "Breitbart",
            "Daily Wire",
            "Daily Caller",
            "Newsmax",
            "OAN",
            "National Review",
            "New York Post",
            "The Federalist",
            "Blaze Media"
        ]
}

random.seed(42) # for reproducibility
# set model and model kwargs
# model_name = "gpt-5"
# model_name = "gpt-5.2-2025-12-11"
models = ["gpt-5-mini"]

model_kwargs = {}

n = 500# number of articles to generate


set_max_concurrent_llm_requests(30) # Set max concurrent requests to 30
# run the async function
payloads = await generate_newspaper_articles(models, n, topics, outlets, system_prompt, user_prompt_template, custom_model_kwargs=model_kwargs)